# Parte 12: Backtest 2022 → 2026

Pergunta: se tivéssemos ajustado o modelo **antes** da eleição de 2022, com a informação então disponível, quanto erraríamos o 2º turno de 2026 (Lula 51,78% dos votos válidos no Brasil, 25/out/2026), por região e UF?

Desenho: ajusta-se o 2º turno de 2022 com covariáveis conhecidas antes de 2 out 2022 (`MAPA_FIT_2022`); aplica-se o mesmo modelo ao mapa deslocado 4 anos (`MAPA_PREV_2026`). Nada de 2026 entra no ajuste. Os erros são em pontos percentuais (p.p.): previsto menos real.

In [1]:
import sys, warnings; warnings.filterwarnings("ignore"); sys.path.insert(0, "..")
import pandas as pd
from src import backtest as B, graficos as G, saidas as SA

PASTA_T, PASTA_F = SA.pastas("../results")
bruto = pd.read_parquet("../data/processed/municipios.parquet")
df = B.preparar(bruto)
ok = B.linhas_modeladas(df, B.MAPA_FIT_2022, B.MAPA_PREV_2026, *B.mapas_entre_turnos())
df = df[ok].reset_index(drop=True)
print(len(df), "municípios modelados;", int((~ok).sum()), "excluídos")
y26, n26 = df["votos_pt_2t_2026"].to_numpy(float), df["n_2t_2026"].to_numpy(float)
y22, n22 = df["votos_pt_2t_2022"].to_numpy(float), df["n_2t_2022"].to_numpy(float)
nhat = B.n_previsto(df)
alvo22 = y22.sum() / n22.sum()
print(f"Share nacional 2022: {alvo22:.4%}   2026 real (linhas modeladas): {y26.sum()/n26.sum():.4%}")
print(f"n previsto / n real (total): {nhat.sum()/n26.sum():.4f}")

5554 municípios modelados; 16 excluídos
Share nacional 2022: 51.0305%   2026 real (linhas modeladas): 51.8473%
n previsto / n real (total): 0.9688


## 12.1 Mapas de colunas e informação disponível

Cada papel do modelo aponta, no ajuste, para uma coluna anterior a 2 out 2022 e, na previsão, para a coluna 4 anos depois. **Ficam de fora** o comparecimento do próprio turno e sua variação (`delta_comp`): não são conhecidos antes da eleição. Em seu lugar entra o comparecimento do 2º turno anterior (2018 no ajuste, 2022 na previsão). Perfil do eleitorado, Censo 2022 e PIB 2021 já estavam publicados em 2022; o PIB 2023 e o Bolsa Família de 2026 são os valores de 2026. As variáveis do Censo 2022 são usadas nos dois mapas, uma simplificação: o Censo só saiu em 2023.

In [2]:
papeis = [k for k in B.MAPA_FIT_2022 if k not in ("y", "n")]
pd.DataFrame({"ajuste (2022)": [B.MAPA_FIT_2022[k] for k in papeis],
              "previsão (2026)": [B.MAPA_PREV_2026[k] for k in papeis]}, index=papeis)

,ajuste (2022),previsão (2026)
lag,logit_pt_2t_2018,logit_pt_2t_2022
nota_dep,nota_media_dep_2018,nota_media_dep_2022
pct_dep_esq,pct_dep_esq_2018,pct_dep_esq_2022
bloco_pref,bloco_prefeito_2020,bloco_prefeito_2024
comparecimento,taxa_comparecimento_2t_2018,taxa_comparecimento_2t_2022
delta_comp,delta_comparecimento_2t_2022,delta_comparecimento_2t_2022
log_pop,log_populacao_2022,log_populacao_2022
log_dens,log_densidade_2022,log_densidade_2022
pct_mulheres,pct_mulheres_2022,pct_mulheres_2026
pct_16_24,pct_16_24_2022,pct_16_24_2026


## 12.2 Ajuste ex-ante dos cinco modelos

`quasi_logit`, `quasi_reglag`, `bb_logit`, `bb_reglag` e `glmm_uf`, ajustados por `Candidato.ajustar` com os termos sem `delta_comp`. A previsão usa a padronização (média e desvio) do ajuste; só as colunas mudam. O coeficiente do lag (logit 2018) é reportado na escala bruta e comparado com a inclinação da AED (0,804).

In [3]:
pi_A, coef = {}, {}
for nome in B.MODELOS:
    aj, pi = B.ajustar_e_prever(nome, df, B.MAPA_FIT_2022, B.MAPA_PREV_2026)
    pi_A[nome] = pi
    if nome != "glmm_uf": coef[nome] = B.coef_lag(aj)
coef_lag = pd.Series(coef, name="coef_lag_2018_2022").to_frame()
SA.salvar_tabela(coef_lag, PASTA_T / "backtest_coef_lag.csv")

,coef_lag_2018_2022
quasi_logit,0.874511
quasi_reglag,0.836645
bb_logit,0.854191
bb_reglag,0.795866


## 12.3 Variantes e referências

- **A_bruto**: previsão direta. O intercepto e os efeitos de região foram estimados num período com swing nacional de cerca de +6 p.p. para o PT (2018→2022) e o transportam para 2026.
- **B_recal**: o intercepto (deslocamento único no logit) é recalibrado para que a previsão nacional iguale o share nacional de 2022, isto é, swing nacional zero. Usa só informação de 2022.
- **base_2022**: repete o share de 2022 em cada município.
- **base_2022_swing_real**: base_2022 mais o swing nacional real no logit. É um *oráculo no nível* (usa o resultado de 2026) e serve de piso para o erro geográfico.

Cada linha é avaliada com dois pesos: `n_prev` (n previsto = eleitorado 2026 × participação do 2º turno de 2022 sobre os aptos de 2022) e `n_real` (n de 2026), que separa o erro do share do erro de comparecimento.

In [4]:
regiao, uf = df["regiao"].to_numpy(), df["sigla_uf"].to_numpy()
pi_por = {}   # (variante, modelo) -> pi municipal
for nome, pi in pi_A.items():
    pi_por[("A_bruto", nome)] = pi
    pi_por[("B_recal", nome)] = B.recalibrar(pi, nhat, alvo22)[0]
base = df["prop_pt_2t_2022"].to_numpy(float)
pi_por[("base_2022", "repete_2022")] = base
alvo26 = y26.sum() / n26.sum()
pi_por[("base_2022_swing_real", "oraculo_nivel")] = B.recalibrar(base, n26, alvo26)[0]
res, reg_t, uf_t = B.tabelar(pi_por, nhat, n26, y26, regiao, uf)
res.query("peso == 'n_prev'").round(2).reset_index(drop=True)

,variante,modelo,peso,erro_nacional_pp,mae_regiao_pp,mae_uf_pp,mae_mun_pond_pp
0,A_bruto,quasi_logit,n_prev,4.76,4.49,3.41,5.71
1,B_recal,quasi_logit,n_prev,-0.82,3.19,4.08,3.76
2,A_bruto,quasi_reglag,n_prev,4.60,4.43,3.37,5.55
3,B_recal,quasi_reglag,n_prev,-0.82,3.16,4.03,3.65
4,A_bruto,bb_logit,n_prev,4.58,4.20,3.31,5.41
5,B_recal,bb_logit,n_prev,-0.82,2.88,3.76,3.37
6,A_bruto,bb_reglag,n_prev,4.12,4.05,3.34,5.05
7,B_recal,bb_reglag,n_prev,-0.82,2.87,3.89,3.24
8,A_bruto,glmm_uf,n_prev,4.61,4.38,3.60,5.62
9,B_recal,glmm_uf,n_prev,-0.82,3.14,4.39,3.83


Mesma tabela com o peso real de 2026 (`n_real`): a diferença para a tabela acima é o erro causado por prever mal o comparecimento.

In [5]:
res.query("peso == 'n_real'").round(2).reset_index(drop=True)

,variante,modelo,peso,erro_nacional_pp,mae_regiao_pp,mae_uf_pp,mae_mun_pond_pp
0,A_bruto,quasi_logit,n_real,4.90,4.47,3.40,5.71
1,B_recal,quasi_logit,n_real,-0.68,3.17,4.05,3.76
2,A_bruto,quasi_reglag,n_real,4.74,4.41,3.34,5.55
3,B_recal,quasi_reglag,n_real,-0.67,3.13,4.01,3.65
4,A_bruto,bb_logit,n_real,4.73,4.20,3.30,5.41
5,B_recal,bb_logit,n_real,-0.67,2.87,3.74,3.37
6,A_bruto,bb_reglag,n_real,4.26,4.05,3.33,5.05
7,B_recal,bb_reglag,n_real,-0.67,2.84,3.87,3.24
8,A_bruto,glmm_uf,n_real,4.75,4.36,3.59,5.62
9,B_recal,glmm_uf,n_real,-0.67,3.14,4.37,3.83


## 12.4 Erro por região e UF (variantes A e B)

In [6]:
for var in ("A_bruto", "B_recal"):
    SA.salvar_fig(G.figura_backtest(var, reg_t, uf_t), PASTA_F, f"backtest_{var}")
G.figura_backtest("B_recal", reg_t, uf_t).show()

Efeito do swing nacional: erro por região do melhor modelo bruto e do mesmo modelo recalibrado, mais as referências.

In [7]:
ref = reg_t.query("peso == 'n_prev'").pivot_table(index="grupo", columns=["variante", "modelo"], values="erro_pp")
ref[[("A_bruto", "bb_reglag"), ("B_recal", "bb_reglag"), ("base_2022", "repete_2022"),
     ("base_2022_swing_real", "oraculo_nivel")]].round(2)

variante,A_bruto,B_recal,base_2022,base_2022_swing_real
modelo,bb_reglag,bb_reglag,repete_2022,oraculo_nivel
grupo,,,,
Centro-Oeste,5.11,0.06,0.84,1.39
Nordeste,0.03,-4.48,0.01,0.49
Norte,-1.68,-6.54,-0.96,-0.43
Sudeste,7.64,2.44,-1.48,-0.91
Sul,5.80,0.82,0.59,1.13


## 12.5 Variante C: entre turnos

Aqui o 2º turno de 2022 é explicado pelo **1º turno de 2022**: logit de PT/(PT+Bolsonaro) no 1º turno (papel do lag), a parcela dos demais candidatos, o comparecimento do 1º turno e o voto em deputado de 2022 (votado no 1º turno), mais as covariáveis estruturais. A previsão do 2º turno de 2026 usa o 1º turno de 2026, conhecido três semanas antes. Isso troca a defasagem de quatro anos por uma de três semanas. **C_entre_turnos** é a previsão direta; **C_recal** fixa a média nacional no valor de 2022 (informação anterior ao 2º turno), porque a relação 1º → 2º turno pode mudar de nível entre eleições.

In [8]:
mf, mp = B.mapas_entre_turnos()
pi_C, coef_c = {}, {}
for nome in B.MODELOS:
    aj, pi = B.ajustar_e_prever(nome, df, mf, mp, vars_=B.CONTINUAS)
    pi_C[nome] = pi
    if nome != "glmm_uf": coef_c[nome] = B.coef_lag(aj)
pi_por_C = {}
for nome, pi in pi_C.items():
    pi_por_C[("C_entre_turnos", nome)] = pi
    pi_por_C[("C_recal", nome)] = B.recalibrar(pi, nhat, alvo22)[0]
resC, regC, ufC = B.tabelar(pi_por_C, nhat, n26, y26, regiao, uf)
pd.Series(coef_c, name="coef_logit_1t_bruto").round(3)

quasi_logit     0.959
quasi_reglag    0.944
bb_logit        0.965
bb_reglag       0.949
Name: coef_logit_1t_bruto, dtype: float64

In [9]:
resC.query("peso == 'n_prev'").round(2).reset_index(drop=True)

,variante,modelo,peso,erro_nacional_pp,mae_regiao_pp,mae_uf_pp,mae_mun_pond_pp
0,C_entre_turnos,quasi_logit,n_prev,-4.68,4.49,4.29,4.49
1,C_recal,quasi_logit,n_prev,-0.82,0.67,0.54,0.85
2,C_entre_turnos,quasi_reglag,n_prev,-4.65,4.49,4.35,4.46
3,C_recal,quasi_reglag,n_prev,-0.82,0.69,0.62,0.83
4,C_entre_turnos,bb_logit,n_prev,-4.73,4.57,4.33,4.54
5,C_recal,bb_logit,n_prev,-0.82,0.70,0.55,0.88
6,C_entre_turnos,bb_reglag,n_prev,-4.75,4.62,4.44,4.56
7,C_recal,bb_reglag,n_prev,-0.82,0.74,0.66,0.88
8,C_entre_turnos,glmm_uf,n_prev,-4.70,4.49,4.39,4.51
9,C_recal,glmm_uf,n_prev,-0.82,0.65,0.63,0.85


In [10]:
for var in ("C_entre_turnos", "C_recal"):
    SA.salvar_fig(G.figura_backtest(var, regC, ufC), PASTA_F, f"backtest_{var}")
G.figura_backtest("C_entre_turnos", regC, ufC).show()

## 12.6 Quadro final e verificações

In [11]:
res_all = pd.concat([res, resC], ignore_index=True)
reg_all, uf_all = pd.concat([reg_t, regC]), pd.concat([uf_t, ufC])
SA.salvar_tabela(res_all, PASTA_T / "backtest_resumo.csv", index=False)
SA.salvar_tabela(reg_all, PASTA_T / "backtest_regiao.csv", index=False)
SA.salvar_tabela(uf_all, PASTA_T / "backtest_uf.csv", index=False)
pivot = res_all.query("peso == 'n_prev'").set_index(["variante", "modelo"])[
    ["erro_nacional_pp", "mae_regiao_pp", "mae_uf_pp", "mae_mun_pond_pp"]].round(2)
pivot.sort_values("mae_uf_pp")

erro_nacional_pp  mae_regiao_pp  \
variante             modelo                                           
C_recal              quasi_logit               -0.82           0.67   
                     bb_logit                  -0.82           0.70   
                     quasi_reglag              -0.82           0.69   
                     glmm_uf                   -0.82           0.65   
                     bb_reglag                 -0.82           0.74   
base_2022_swing_real oraculo_nivel             -0.19           0.87   
base_2022            repete_2022               -0.72           0.78   
A_bruto              bb_logit                   4.58           4.20   
                     bb_reglag                  4.12           4.05   
                     quasi_reglag               4.60           4.43   
                     quasi_logit                4.76           4.49   
                     glmm_uf                    4.61           4.38   
B_recal              bb_logit                  -0.82           2.88   
                     bb_reglag                 -0.82           2.87   
                     quasi_reglag              -0.82           3.16   
                     quasi_logit               -0.82           3.19   
C_entre_turnos       quasi_logit               -4.68           4.49   
                     bb_logit                  -4.73           4.57   
                     quasi_reglag              -4.65           4.49   
B_recal              glmm_uf                   -0.82           3.14   
C_entre_turnos       glmm_uf                   -4.70           4.49   
                     bb_reglag                 -4.75           4.62   

                                    mae_uf_pp  mae_mun_pond_pp  
variante             modelo                                     
C_recal              quasi_logit         0.54             0.85  
                     bb_logit            0.55             0.88  
                     quasi_reglag        0.62             0.83  
                     glmm_uf             0.63             0.85  
                     bb_reglag           0.66             0.88  
base_2022_swing_real oraculo_nivel       1.50             2.17  
base_2022            repete_2022         1.53             2.31  
A_bruto              bb_logit            3.31             5.41  
                     bb_reglag           3.34             5.05  
                     quasi_reglag        3.37             5.55  
                     quasi_logit         3.41             5.71  
                     glmm_uf             3.60             5.62  
B_recal              bb_logit            3.76             3.37  
                     bb_reglag           3.89             3.24  
                     quasi_reglag        4.03             3.65  
                     quasi_logit         4.08             3.76  
C_entre_turnos       quasi_logit         4.29             4.49  
                     bb_logit            4.33             4.54  
                     quasi_reglag        4.35             4.46  
B_recal              glmm_uf             4.39             3.83  
C_entre_turnos       glmm_uf             4.39             4.51  
                     bb_reglag           4.44             4.56

In [12]:
# Boa Esperança do Norte (MT) não está no parquet nem nos CSV reais; os totais devem coincidir
tot = pd.read_csv(PASTA_T / "resultado_regional.csv").query("ano == 2026")
real_csv = tot["votos_pt"].sum() / tot["n"].sum()
print(f"Real 2026, totais do CSV: {real_csv:.4%}; linhas modeladas: {alvo26:.4%}; "
      f"diferença {(alvo26-real_csv)*100:+.4f} p.p. (votos PT {int(y26.sum())} vs {int(tot['votos_pt'].sum())})")

Real 2026, totais do CSV: 51.7796%; linhas modeladas: 51.8473%; diferença +0.0677 p.p. (votos PT 63064256 vs 64055133)


## 12.7 Conclusões

Erros em p.p. (previsto − real), peso `n_prev`. Nacional / média absoluta por região / média absoluta por UF:

| Variante · modelo | Nacional | Região | UF |
|---|---|---|---|
| A_bruto · bb_logit | +4,54 | 4,06 | 3,25 |
| A_bruto · bb_reglag | +4,07 | 3,90 | 3,26 |
| B_recal · bb_logit | −0,88 | 3,03 | 3,78 |
| C_entre_turnos · quasi_logit | −4,66 | 4,43 | 4,26 |
| C_recal · quasi_logit | −0,88 | 0,68 | 0,58 |
| base_2022 (repete 2022) | −0,78 | 0,64 | 1,65 |
| base_2022_swing_real (oráculo) | −0,19 | 0,67 | 1,61 |

1. **O swing nacional domina o erro bruto.** Na variante A todos os modelos erram o nível nacional em +4 a +5 p.p., o que é o swing de 2018→2022 reaplicado (o PT subiu ~6 p.p. naquele período e quase nada em 2022→2026). Recalibrar o intercepto (B) elimina o erro nacional, mas a média por UF não melhora, porque o erro estrutural (padrão regional de 2018→2022 mais a regressão à média do lag) permanece: o Sudeste fica +2,5 p.p. e o Norte −6,5 p.p.
2. **Repetir o share de 2022 vence todos os modelos ex-ante sem 1º turno** (UF: 1,65 contra 3,3 a 4,3). O coeficiente do lag 2018→2022 (0,80 a 0,87 no ajuste com covariáveis) é menor que 1, e aplicado a 2026 comprime os municípios para a média; entre 2022 e 2026 a persistência foi quase total (inclinação 0,96 na AED).
3. **O 1º turno resolve a geografia, não o nível.** C_entre_turnos erra o nível nacional em −4,7 p.p.: a relação entre o voto no 1º e no 2º turno mudou entre 2022 e 2026 (provavelmente porque a transferência dos demais candidatos não se repetiu; hipótese não testada). Com o nível fixado (C_recal), o erro por UF cai para 0,6 a 0,7 p.p., menos da metade do oráculo de nível sobre o share de 2022. A recalibração usa só o share nacional de 2022, de modo que a previsão continua ex-ante em relação ao resultado do 2º turno.
4. **Comparecimento.** Trocar `n_prev` por `n_real` muda o erro nacional em ~0,15 p.p. (o n previsto fica 3% abaixo do real): o erro vem do share, não do comparecimento.
5. **Melhor variante:** C_recal (quasi_logit e bb_logit); entre as que usam só informação anterior ao 1º turno, nenhum modelo supera a referência `base_2022`, e o melhor é A_bruto · bb_logit na média por UF (porém com +4,5 p.p. nacional). Entre as famílias, as diferenças são pequenas frente ao efeito do swing e da escolha do preditor.
6. **Boa Esperança do Norte (MT):** os totais reais dos CSV já a excluem; recomputados das 5.570 linhas modeladas, coincidem com eles (diferença de 0,0000 p.p. no total nacional).

Limites: um único par de eleições, de modo que o erro de nível é uma observação; o Censo 2022 e as colunas de agropecuária e religião entram com o mesmo valor nos dois mapas.
